In [ ]:
def plot_acquisition_heatmap(
    X,                   # shape (n_samples, 2): queried points
    acquisition_map,     # shape (grid_size**2,): acquisition values
    X_grid,              # shape (grid_size**2, 2): grid points
    next_query=None,     # shape (2,), optional: next query point
    title='Acquisition Heatmap',
    cmap='viridis',
    color_range=(-3, 1), # default color scale for consistency
    save_path=None       # optional: path to save plot
):
    grid_size = int(np.sqrt(len(acquisition_map)))
    acquisition_2d = acquisition_map.reshape(grid_size, grid_size)

    # Compute extent from grid
    x_min, x_max = X_grid[:, 0].min(), X_grid[:, 0].max()
    y_min, y_max = X_grid[:, 1].min(), X_grid[:, 1].max()
    extent = [x_min, x_max, y_min, y_max]

    plt.figure(figsize=(8, 6))
    im = plt.imshow(
        acquisition_2d,
        origin='lower',
        extent=extent,
        cmap=cmap,
        aspect='auto',
        vmin=color_range[0],
        vmax=color_range[1]
    )
    plt.colorbar(im, label='Acquisition Value')

    # Queried points
    plt.scatter(X[:, 0], X[:, 1], c='white', edgecolors='black', label='Queried Points')

    # Next query
    if next_query is not None:
        plt.plot(next_query[0], next_query[1], 'r*', markersize=14, label='Next Query')

    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.legend()
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
    else:
        plt.show()

In [ ]:
# Example DataFrame
"""def add_points_to_df(df, results):"""
"""
    Appends rows to df using:
    - x values from each result
    - an empty string for 'yield'
    - the method name as 'source'
    """
"""    for result in results:
        point = list(result['x'])
        row = point + [np.nan] + ['week-x']
        df.loc[len(df)] = row """

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

def plot_output_points(df, scale_factor=1.0, yield_scale_factor=1.0, nan_marker='o'):
    """
    Plots the second-to-last column as 'Yield' and all other columns (except the last)
    as scaled variables, each with a unique color and no connecting lines.

    Parameters:
    - df: pandas DataFrame
    - scale_factor: numeric value to multiply non-yield variables (default is 1000)
    - nan_marker: marker style for points where 'yield' is missing (default is 'o')
    """
    columns = df.columns
    yield_col = columns[-2]
    other_cols = [col for col in df.columns if col != yield_col and col != columns[-1]]

    # Clean and convert yield column
    df[yield_col] = df[yield_col].replace('', np.nan)

    # Generate distinct colors
    cmap = cm.get_cmap('tab10', len(other_cols))

    # Plot yield values (non-NaN)
    if df[yield_col].notna().any():
        plt.plot(df[yield_col].astype(float) * yield_scale_factor, marker='x', linestyle='none', label=yield_col, color='black')

    # Plot missing yield points using nan_marker
    missing_yield_indices = df[yield_col].isna()
    if missing_yield_indices.any():
        plt.plot(df.index[missing_yield_indices], [0] * missing_yield_indices.sum(),
                 marker=nan_marker, linestyle='none', label='missing yield', color='gray')

    # Plot other variables scaled by scale_factor
    for i, col in enumerate(other_cols):
        plt.plot(df[col].astype(float) * scale_factor, marker='.', linestyle='none', label=col, color=cmap(i))

    plt.xlabel('Index')
    plt.ylabel('Output y')
    plt.title('Output Curve')
    plt.grid(True)
    plt.legend()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import numpy as np

def plot_surrogate_generic(X_grid, y_pred, 
                           X_filtered=None, 
                           bounds=None,
                           candidates=None,
                           candidate_labels=None,
                           title="NN Surrogate"):
    """
    Plot surrogate predictions with optional filtered points and candidate queries.

    Parameters
    ----------
    X_grid : array of shape (N, d)
        Grid points (can be full or filtered)
    y_pred : array of shape (N,)
        Surrogate predictions
    X_filtered : array of shape (M, d), optional
        Points actually evaluated (highlighted separately)
    bounds : list of (min, max) for each dimension, optional
        Defines the bounding box of the unfiltered grid
    candidates : array of shape (K, d), optional
        Candidate query points selected via Bayesian optimisation
    candidate_labels : list of str, optional
        Labels for candidate points (e.g., iteration index or acquisition score)
    title : str
        Plot title
    """
    d = X_grid.shape[1]

    if d == 1:
        plt.figure(figsize=(6,4))
        plt.plot(X_grid[:,0], y_pred, label="NN surrogate")

        if X_filtered is not None:
            plt.scatter(X_filtered[:,0], 
                        np.interp(X_filtered[:,0], X_grid[:,0], y_pred),
                        color="red", label="Filtered points")

        if candidates is not None:
            plt.scatter(candidates[:,0],
                        np.interp(candidates[:,0], X_grid[:,0], y_pred),
                        color="blue", marker="*", s=120, label="Candidate queries")
            if candidate_labels is not None:
                for i, pt in enumerate(candidates):
                    plt.text(pt[0], 
                             np.interp(pt[0], X_grid[:,0], y_pred),
                             candidate_labels[i],
                             color="blue", fontsize=9, ha="left")

        if bounds is not None:
            x_min, x_max = bounds[0]
            plt.axvline(x_min, color="black", linestyle="--", label="Bounds")
            plt.axvline(x_max, color="black", linestyle="--")

        plt.xlabel("x")
        plt.ylabel("f(x)")
        plt.title(title)
        plt.legend()
        plt.show()

    elif d == 2:
        plt.figure(figsize=(6,5))
        scatter = plt.scatter(X_grid[:,0], X_grid[:,1], 
                              c=y_pred, cmap="viridis", s=20)
        plt.colorbar(scatter, label="Model prediction")

        if X_filtered is not None:
            plt.scatter(X_filtered[:,0], X_filtered[:,1], 
                        c="red", edgecolor="black", s=40, label="Filtered points")

        if candidates is not None:
            plt.scatter(candidates[:,0], candidates[:,1],
                        c="blue", marker="*", s=120, label="Candidate queries")
            if candidate_labels is not None:
                for i, pt in enumerate(candidates):
                    plt.text(pt[0], pt[1], candidate_labels[i],
                             color="blue", fontsize=9, ha="left")

        if bounds is not None:
            x_min, x_max = bounds[0]
            y_min, y_max = bounds[1]
            rect_x = [x_min, x_max, x_max, x_min, x_min]
            rect_y = [y_min, y_min, y_max, y_max, y_min]
            plt.plot(rect_x, rect_y, color="black", linewidth=2, label="Bounds")

        plt.xlabel("x1")
        plt.ylabel("x2")
        plt.title(title)
        plt.legend()
        plt.show()

    else:
        i, j = 0, 1
        plt.figure(figsize=(6,5))
        scatter = plt.scatter(X_grid[:,i], X_grid[:,j], c=y_pred, cmap="viridis", s=20)
        plt.colorbar(scatter, label="Model prediction")

        if X_filtered is not None:
            plt.scatter(X_filtered[:,i], X_filtered[:,j], 
                        c="red", edgecolor="black", s=40, label="Filtered points")

        if candidates is not None:
            plt.scatter(candidates[:,i], candidates[:,j],
                        c="blue", marker="*", s=120, label="Candidate queries")
            if candidate_labels is not None:
                for idx, pt in enumerate(candidates):
                    plt.text(pt[i], pt[j], candidate_labels[idx],
                             color="blue", fontsize=9, ha="left")

        if bounds is not None:
            x_min, x_max = bounds[i]
            y_min, y_max = bounds[j]
            rect_x = [x_min, x_max, x_max, x_min, x_min]
            rect_y = [y_min, y_min, y_max, y_max, y_min]
            plt.plot(rect_x, rect_y, color="black", linewidth=2, label="Bounds")

        plt.xlabel(f"x{i}")
        plt.ylabel(f"x{j}")
        plt.title(f"{title} (dims {i},{j})")
        plt.legend()
        plt.show()

def plot_loss_curve(history, title="Training Loss Curve"):
    plt.figure(figsize=(6,4))
    plt.plot(history["train_losses"], label="Training loss")   # <-- dict access
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.show()

def plot_weight_evolution(history, layer_name="model.0.weight", stat="mean", title="Weight Evolution"):
    """
    Plot weight evolution after training using the history dict.
    
    Parameters
    ----------
    history : dict returned by SurrogateTrainer.fit
    layer_name : str, name of the layer to track (e.g. 'model.0.weight')
    stat : str, which statistic to plot ('mean' or 'std')
    title : str, plot title
    """
    if history["weights"] is None:
        raise ValueError("No weight log found. Run fit with log_weights=True.")

    values = [epoch_stats[layer_name][stat] for epoch_stats in history["weights"]]

    plt.figure(figsize=(6,4))
    plt.plot(values)
    plt.xlabel("Epoch")
    plt.ylabel(f"{stat} value")
    plt.title(title)
    plt.show()

In [ ]:
def extract_candidates_and_labels(query_results, include_score=False):
    """
    Convert query_results into candidate points and labels for plotting.
    
    Parameters
    ----------
    query_results : list of dict
        Logged candidates from log_query_candidate
    include_score : bool
        Whether to append acquisition score to labels
    
    Returns
    -------
    candidates : np.ndarray of shape (K, d)
    labels : list of str
    """
    candidates = np.array([entry['x'] for entry in query_results])
    labels = []
    for entry in query_results:
        if entry['method'] in ['PI', 'EI']:
            label = f"{entry['method']} (xi={entry['xi']})"
        else:  # UCB
            label = f"{entry['method']} (kappa={entry['kappa']})"
        if include_score:
            label += f", score={entry['score']:.3f}"
        labels.append(label)
    return candidates, labels